# Clasificador por casilla v2 — DESDE CERO (dataset limpio)

Entrena **(b) el clasificador de 13 clases**: las 12 piezas + `vacia`.

| | |
|---|---|
| Modelo | `mobilenet_v3_small`, `weights=IMAGENET`, **sin congelar** |
| Entrada | casilla rectificada 100x100 -> 128x128 |
| Paquete | `clasificador_v2_clean.zip` (~474 MB) en `Mi unidad/ajedrez_v2/` |
| Train | 139,200 casillas de 2,175 tableros |
| Valid | 17,408 casillas de 272 tableros |
| Test | 17,408 casillas de 272 tableros |
| Épocas | 20 con `AMP` y pesos de clase por frecuencia inversa |

**Por qué este modelo:** YOLO detecta cajas e ignora que el tablero es una rejilla
conocida de 64 casillas. Aquí se rectifica el tablero, se recortan las 64 casillas
y se clasifica cada una. El reparto se hace **por tablero, nunca por casilla**:
las 64 casillas de una misma foto caen siempre en el mismo split.

Los recortes reales del `valid` y del `test` son **los mismos** que usa el
detector (mismo reparto por imagen fuente): la comparación de T7 se mide sobre
las mismas fotos.

Antes de ejecutar: **Entorno de ejecución > Cambiar tipo de entorno > T4 GPU**.

## 1. Verificar la GPU

In [ ]:
#@title 1. Verificar GPU y versiones
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch
print("PyTorch:", torch.__version__)
print("CUDA de PyTorch:", torch.version.cuda)
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("SIN GPU: ve a Entorno de ejecucion > Cambiar tipo de entorno > T4 GPU")

## 2. Dependencias

`torch`, `torchvision` y `Pillow` ya vienen en Colab. La celda sólo instala si
falta algo y **nunca** reinstala `torch`: cambiarlo rompe la GPU de la sesión.

In [ ]:
#@title 2. Comprobar dependencias sin tocar torch
import importlib.util
import subprocess
import sys

for paquete, modulo in (("torchvision", "torchvision"), ("pillow", "PIL")):
    if importlib.util.find_spec(modulo) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete],
                       check=True)

import torch
import torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__)

## 3. Traer el paquete de datos

El zip trae:

```
casillas/<clase>/<archivo>.jpg        191,072 recortes (13 clases)
                                      100x100 compuesto/reales + 128x128 aumentadas
split/{train,valid,test}.txt          una ruta RELATIVA por linea
labels/clases.txt                     las 13 clases en orden
MANIFEST.txt                          de donde sale cada casilla
```

La lista de cada split es la que manda: no hay `ImageFolder` ni carpetas por
split. Se descomprime en `/content` (disco local), no en Drive.

In [ ]:
#@title 3. Descomprimir clasificador_v2_clean.zip y contar por split
import shutil
import zipfile
from pathlib import Path

USAR_DRIVE = True
ZIP_EN_DRIVE = Path("/content/drive/MyDrive/ajedrez_v2/clasificador_v2_clean.zip")
DESTINO = Path("/content/ajedrez_v2/clasificador")
ESPERADO = {"train": 157280, "valid": 16896, "test": 16896}

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    origen = ZIP_EN_DRIVE
else:
    from google.colab import files
    subidos = files.upload()
    origen = Path(next(iter(subidos)))

if not origen.exists():
    raise FileNotFoundError(
        f"No encuentro {origen}. Sube clasificador_v2_clean.zip a "
        "Mi unidad/ajedrez_v2/ o pon USAR_DRIVE = False."
    )

if DESTINO.exists():
    shutil.rmtree(DESTINO)
DESTINO.mkdir(parents=True)

with zipfile.ZipFile(origen) as zf:
    zf.extractall(DESTINO)

print("Contenido:", sorted(p.name for p in DESTINO.iterdir()))
for split, esperado in ESPERADO.items():
    n = sum(1 for linea in (DESTINO / "split" / f"{split}.txt").read_text().splitlines()
            if linea.strip())
    print(f"  {split:5s} rutas={n:6d} (esperado {esperado:6d}) "
          f"{'OK' if n == esperado else 'REVISAR'}")

CLASES = [l.strip() for l in (DESTINO / "labels" / "clases.txt").read_text().splitlines()
          if l.strip()]
print("Clases (13):", CLASES)
assert len(CLASES) == 13 and CLASES[12] == "vacia", "labels/clases.txt no tiene 13 clases"

PROYECTO = Path("/content/drive/MyDrive/ajedrez_v2/clasificador")
PROYECTO.mkdir(parents=True, exist_ok=True)
print("Salida:", PROYECTO)

## 4. Dataset, transformaciones y pesos de clase

`Dataset` mínimo que lee la lista de rutas del `.txt` y saca la clase del
**nombre de la carpeta** (`casillas/<clase>/archivo.jpg`).

- `Resize(128)` + normalización de ImageNet (el modelo viene de ahí).
- Sólo en train, `ColorJitter` suave y `RandomRotation` de ±12°: las fichas
  físicas se pueden colocar giradas.
- **Pesos de clase** = frecuencia inversa sobre el split de train. `vacia` es el
  55.9 % de las casillas; sin el peso el modelo aprende a decir "vacia" y listo.

In [ ]:
#@title 4. Datasets, loaders y pesos de clase
from collections import Counter
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

RAIZ = DESTINO
TAMANO = 128
BATCH = 128
EPOCAS = 20
LR = 1e-3
SEMILLA = 42
NUM_WORKERS = 4

torch.manual_seed(SEMILLA)

TRANSFORM_TRAIN = transforms.Compose([
    transforms.Resize((TAMANO, TAMANO)),
    transforms.RandomRotation(12),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
TRANSFORM_EVAL = transforms.Compose([
    transforms.Resize((TAMANO, TAMANO)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class Casillas(Dataset):
    """Casillas de un split, leidas desde su .txt (rutas relativas al zip)."""

    def __init__(self, raiz: Path, txt: Path, clases: list, transform):
        self.raiz = raiz
        self.clases = clases
        self.transform = transform
        self.rutas = [l.strip() for l in txt.read_text(encoding="utf-8").splitlines()
                      if l.strip()]

    def __len__(self) -> int:
        return len(self.rutas)

    def __getitem__(self, indice: int):
        relativa = self.rutas[indice]
        clase = relativa.split("/")[1]          # casillas/<clase>/<archivo>.jpg
        imagen = Image.open(self.raiz / relativa).convert("RGB")
        return self.transform(imagen), self.clases.index(clase)


conjuntos = {
    split: Casillas(RAIZ, RAIZ / "split" / f"{split}.txt", CLASES,
                    TRANSFORM_TRAIN if split == "train" else TRANSFORM_EVAL)
    for split in ("train", "valid", "test")
}
cargadores = {
    "train": DataLoader(conjuntos["train"], batch_size=BATCH, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True),
    "valid": DataLoader(conjuntos["valid"], batch_size=BATCH, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True),
    "test": DataLoader(conjuntos["test"], batch_size=BATCH, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=True),
}

conteo_train = Counter(ruta.split("/")[1] for ruta in conjuntos["train"].rutas)
faltan = [c for c in CLASES if conteo_train[c] == 0]
if faltan:
    raise SystemExit(f"Clases sin ejemplos en train: {faltan}. Revisa split/train.txt.")

totales = sum(conteo_train.values())
pesos = torch.tensor(
    [totales / (len(CLASES) * conteo_train[c]) for c in CLASES], dtype=torch.float32
)
pesos = pesos / pesos.mean()   # media 1: no mueve la escala del learning rate

print(f"{'clase':12s} {'train':>8s} {'%':>7s} {'peso':>6s}")
for clase, peso in zip(CLASES, pesos.tolist()):
    n = conteo_train[clase]
    print(f"{clase:12s} {n:8d} {100 * n / totales:6.2f}% {peso:6.2f}")
print(f"\nvacia = {100 * conteo_train['vacia'] / totales:.2f}% del train")
print("items:", {k: len(v) for k, v in conjuntos.items()})

## 5. Modelo, optimizador y función de pérdida

`mobilenet_v3_small` con `weights=IMAGENET` y **todas** las capas entrenables:
no se congela `features`, porque el dominio (fichas sobre tablero impreso) no se
parece a ImageNet. `AdamW(lr=1e-3)` + cosine, `CrossEntropyLoss(weight=...)` y
**AMP** (`torch.amp.autocast` + `GradScaler`): en una T4 casi duplica el
rendimiento por época sin cambiar el resultado.

In [ ]:
#@title 5. Crear el modelo (ImageNet, sin congelar) y AMP
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import models

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
USAR_AMP = DISPOSITIVO == "cuda"


def crear_modelo(n_clases: int) -> nn.Module:
    try:
        red = models.mobilenet_v3_small(
            weights=models.MobileNet_V3_Small_Weights.DEFAULT
        )
    except (AttributeError, TypeError):          # torchvision viejo
        red = models.mobilenet_v3_small(pretrained=True)
    for parametro in red.features.parameters():  # SIN congelar
        parametro.requires_grad = True
    entrada = red.classifier[3].in_features
    red.classifier[3] = nn.Linear(entrada, n_clases)
    return red


modelo = crear_modelo(len(CLASES)).to(DISPOSITIVO)
optimizador = AdamW(modelo.parameters(), lr=LR, weight_decay=1e-4)
planificador = CosineAnnealingLR(optimizador, T_max=EPOCAS)
criterio = nn.CrossEntropyLoss(weight=pesos.to(DISPOSITIVO))
escalador = torch.amp.GradScaler("cuda", enabled=USAR_AMP)

n_parametros = sum(p.numel() for p in modelo.parameters())
entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print("Dispositivo:", DISPOSITIVO, "| AMP:", USAR_AMP)
print(f"Parametros: {n_parametros / 1e6:.2f} M ({entrenables / 1e6:.2f} M entrenables)")
print("Salida:", modelo.classifier[3].out_features, "clases")

## 6. Entrenar (20 épocas)

Se guarda en Drive cada vez que `valid` mejora. Si Colab se desconecta, el
`best_clasificador.pt` sigue en tu unidad.

In [ ]:
#@title 6. Entrenar con AMP
import copy
import time

def evaluar(red, cargador):
    red.eval()
    perdida_total = 0.0
    aciertos = 0
    vistos = 0
    with torch.no_grad():
        for imagenes, etiquetas in cargador:
            imagenes = imagenes.to(DISPOSITIVO, non_blocking=True)
            etiquetas = etiquetas.to(DISPOSITIVO, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=USAR_AMP):
                salida = red(imagenes)
                perdida = criterio(salida, etiquetas)
            perdida_total += perdida.item() * len(etiquetas)
            aciertos += (salida.argmax(1) == etiquetas).sum().item()
            vistos += len(etiquetas)
    return perdida_total / max(vistos, 1), aciertos / max(vistos, 1)


SALIDA = PROYECTO
SALIDA.mkdir(parents=True, exist_ok=True)
mejor_acc = 0.0
historial = []

for epoca in range(1, EPOCAS + 1):
    arranque = time.time()
    modelo.train()
    perdida_total = 0.0
    aciertos = 0
    vistos = 0
    for imagenes, etiquetas in cargadores["train"]:
        imagenes = imagenes.to(DISPOSITIVO, non_blocking=True)
        etiquetas = etiquetas.to(DISPOSITIVO, non_blocking=True)
        optimizador.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=USAR_AMP):
            salida = modelo(imagenes)
            perdida = criterio(salida, etiquetas)
        escalador.scale(perdida).backward()
        escalador.step(optimizador)
        escalador.update()
        perdida_total += perdida.item() * len(etiquetas)
        aciertos += (salida.argmax(1) == etiquetas).sum().item()
        vistos += len(etiquetas)
    planificador.step()

    perdida_train = perdida_total / max(vistos, 1)
    acc_train = aciertos / max(vistos, 1)
    perdida_val, acc_val = evaluar(modelo, cargadores["valid"])
    segundos = time.time() - arranque
    historial.append((epoca, perdida_train, acc_train, perdida_val, acc_val))
    print(f"epoca {epoca:2d}/{EPOCAS}  loss={perdida_train:.4f} "
          f"train={acc_train:.3f} valid={acc_val:.3f}  ({segundos:.0f}s)")

    if acc_val > mejor_acc:
        mejor_acc = acc_val
        torch.save({
            "modelo": copy.deepcopy(modelo.state_dict()),
            "clases": CLASES,
            "epoca": epoca,
            "acc_valid": acc_val,
            "tamano": TAMANO,
        }, SALIDA / "best_clasificador.pt")
        print(f"  -> mejor hasta ahora (valid={acc_val:.3f}), guardado en Drive")

print()
print(f"Mejor accuracy en valid: {mejor_acc:.4f}")
print("Pesos:", SALIDA / "best_clasificador.pt")

## 7. Matriz de confusión por clase

El accuracy global esconde el problema: `vacia` es el 56 % de las casillas. Si el
modelo sólo acierta `vacia`, el número se ve bien y el pipeline falla. Aquí se ve
clase por clase, y el par `caballo_n <-> alfil_n` en particular.

In [ ]:
#@title 7. Matriz de confusion y reporte por clase
import numpy as np

def matriz_confusion(red, cargador, n_clases: int) -> np.ndarray:
    matriz = np.zeros((n_clases, n_clases), dtype=np.int64)
    red.eval()
    with torch.no_grad():
        for imagenes, etiquetas in cargador:
            salida = red(imagenes.to(DISPOSITIVO)).argmax(1).cpu().numpy()
            for real, pred in zip(etiquetas.numpy(), salida):
                matriz[real, pred] += 1
    return matriz


if not (SALIDA / "best_clasificador.pt").exists():
    raise SystemExit("No hay best_clasificador.pt: ejecuta la celda 6 primero")

pesos_guardados = torch.load(SALIDA / "best_clasificador.pt", map_location=DISPOSITIVO)
modelo.load_state_dict(pesos_guardados["modelo"])

for split in ("valid", "test"):
    matriz = matriz_confusion(modelo, cargadores[split], len(CLASES))
    total = matriz.sum()
    aciertos = np.trace(matriz)
    print(f"== {split} ==")
    print(f"   accuracy global: {aciertos / total:.4f}  ({aciertos}/{total})")
    print(f"   {'clase':12s} {'n':>7s} {'prec':>6s} {'recall':>6s}")
    for i, clase in enumerate(CLASES):
        n = matriz[i].sum()
        if n == 0:
            print(f"   {clase:12s} {0:7d}   (sin ejemplos)")
            continue
        recall = matriz[i, i] / n
        predichos = matriz[:, i].sum()
        precision = matriz[i, i] / predichos if predichos else 0.0
        print(f"   {clase:12s} {n:7d} {precision:6.3f} {recall:6.3f}")

    errores = []
    for real in range(len(CLASES)):
        for pred in range(len(CLASES)):
            if real != pred and matriz[real, pred] > 0:
                errores.append((int(matriz[real, pred]), CLASES[real], CLASES[pred]))
    errores.sort(reverse=True)
    print("   confusiones mas frecuentes:")
    for cantidad, real, pred in errores[:8]:
        print(f"     {real:12s} -> {pred:12s} {cantidad}")
    print()

## 8. Guardar el modelo final en Drive

`best_clasificador.pt` guarda el `state_dict`, la lista de clases y el tamaño de
entrada. Para usarlo hay que reconstruir `mobilenet_v3_small` y cargar el
`state_dict`: el archivo no trae la arquitectura.

In [ ]:
#@title 8. Verificar que el modelo se puede recargar y listar Drive
import shutil

SALIDA_ENTREGA = PROYECTO / "entrega"
SALIDA_ENTREGA.mkdir(parents=True, exist_ok=True)

shutil.copy2(PROYECTO / "best_clasificador.pt",
             SALIDA_ENTREGA / "best_clasificador.pt")
(SALIDA_ENTREGA / "clases.txt").write_text("\n".join(CLASES) + "\n",
                                           encoding="utf-8")

comprobacion = crear_modelo(len(CLASES)).to(DISPOSITIVO)
comprobacion.load_state_dict(torch.load(SALIDA_ENTREGA / "best_clasificador.pt",
                                        map_location=DISPOSITIVO)["modelo"])
comprobacion.eval()
print("Recarga OK:", SALIDA_ENTREGA / "best_clasificador.pt")
for p in sorted(SALIDA_ENTREGA.iterdir()):
    print(f"  {p.name:28s} {p.stat().st_size / 1024:8.1f} KB")

## Cómo entra esto en el pipeline v2

1. **Detector YOLO** (`best.pt` del otro notebook) — segunda opinión.
2. **Rectificación** del tablero con las 4 esquinas ArUco -> 800x800.
3. **Clasificador por casilla** (`best_clasificador.pt`): 64 recortes, 13 clases.
4. **Filtro de legalidad** con `python-chess` sobre el FEN construido.

## Si algo falla

| Síntoma | Causa | Solución |
|---|---|---|
| `SIN GPU` | Entorno sin GPU | Entorno de ejecución > Cambiar tipo de entorno > T4 GPU |
| `No encuentro ...clasificador_v2_clean.zip` | Ruta de Drive distinta | Revisa `ZIP_EN_DRIVE` o pon `USAR_DRIVE = False` |
| Descomprimir tarda | 174 mil archivos pequeños | Es normal: 4-6 minutos |
| `CUDA out of memory` | `BATCH` alto | Baja `BATCH` a 64 |
| `accuracy` alta pero el pipeline falla | El modelo dice `vacia` a todo | Mira el recall por clase |
| DataLoader lento | `NUM_WORKERS` bajo | Sube `NUM_WORKERS` a 8 |